# Toots abrufen

In [1]:
from bs4 import BeautifulSoup
import re
import mastodon
import time
from credentials import MASTODON_INSTANCE
from credentials import ACCESS_TOKEN

HASHTAG = "hamburg" # Ohne #
TOOTS = 1000        # Anzahl der zu ladenden Toots
FILENAME = 'hamburg.txt' # Datei für extrahierte Toots

# HTML Code und URLs entfernen
def remove_html(html_content):
    soup = BeautifulSoup(html_content, 'html.parser')
    # HTML
    text = soup.get_text()
    # URLs
    url_pattern = re.compile(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\(\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+')
    text = url_pattern.sub('', text)
    # Leerzeichen und Zeilenumbrüche
    text = re.sub(r'\s+', ' ', text).strip()
    # Leerzeichen zwischen Klein- und Großbuchstaben einfügen
    text = re.sub(r'([a-z])([A-Z])', r'\1 \2', text)
    # Leerzeichen nach Satzzeichen einfügen
    text = re.sub(r'\.(?=\w)', '. ', text)
    text = re.sub(r',(?=\w)', ', ', text)
    return text

# Toots in Datei schreiben
def save_list_to_file(data_list, filename):
    with open(filename, 'w', encoding='utf-8') as f:
        for item in data_list:
            f.write(item + '\n')

# Mastodon initialisieren
try:
    masto = mastodon.Mastodon(
        access_token = ACCESS_TOKEN,
        api_base_url = MASTODON_INSTANCE
    )

except Exception as e:
    print(f"Fehler bei der Mastodon Verbindung: {e}")
    exit()

# Toots laden
all_toots = []
max_id = None
limit = 40  # Limit pro Abfrage
total_to_fetch = TOOTS  # Anzahl der Toots
toots_fetched = 0

try:
    print("Mastodon Toots laden...")
    toots = masto.timeline_hashtag(
        hashtag=HASHTAG,
        limit=limit
    )

    if not toots:
        print("Keine Toots gefunden.")
        exit()

    all_toots.extend(toots)
    toots_fetched += len(toots)
    max_id = toots[-1]['id']
    rounds = 1
    print("Durchgang 1 fertig")

    # Weitere Durchgänge
    while len(all_toots) < total_to_fetch:
        if toots_fetched >= 280:    # Rate Limit beträgt 300 pro 5 Minuten
            print("Rate Limit erreicht. Warte 5 Minuten...")
            time.sleep(300)
            toots_fetched = 0

        more_toots = masto.timeline_hashtag(
            hashtag=HASHTAG,
            limit=limit,
            max_id=max_id
        )

        if not more_toots:
            break

        all_toots.extend(more_toots)
        toots_fetched += len(more_toots)
        max_id = more_toots[-1]['id']
        rounds += 1
        print(f"Durchgang {rounds} fertig")

        if len(all_toots) >= total_to_fetch:
            all_toots = all_toots[:total_to_fetch]
            print(f"{len(all_toots)} Toots in {rounds} Durchgängen geladen")
            break
except Exception as e:
    print(f"Fehler in Mastodon Abfrage: {e}")
    exit()

# Daten verarbeiten und in Datei schreiben
try:
    data = []
    for toot in all_toots:
        data.append({
            #'id': toot['id'],
            'content': remove_html(toot['content']),
            #'created_at': toot['created_at'],
            #'author': toot['account']['acct'],
            #'url': toot['url']
        })

    content_list = []
    for item in data:
        content_list.append(item['content'])
    save_list_to_file(content_list, FILENAME)
    print(f"Toots in {FILENAME} geschrieben")

except Exception as e:
    print(f"Fehler beim Speichern: {e}")
    exit()

Mastodon Toots laden...
Durchgang 1 fertig
Durchgang 2 fertig
Durchgang 3 fertig
Durchgang 4 fertig
Durchgang 5 fertig
Durchgang 6 fertig
Durchgang 7 fertig
Rate Limit erreicht. Warte 5 Minuten...
Durchgang 8 fertig
Durchgang 9 fertig
Durchgang 10 fertig
Durchgang 11 fertig
Durchgang 12 fertig
Durchgang 13 fertig
Durchgang 14 fertig
Rate Limit erreicht. Warte 5 Minuten...
Durchgang 15 fertig
Durchgang 16 fertig
Durchgang 17 fertig
Durchgang 18 fertig
Durchgang 19 fertig
Durchgang 20 fertig
Durchgang 21 fertig
Rate Limit erreicht. Warte 5 Minuten...
Durchgang 22 fertig
Durchgang 23 fertig
Durchgang 24 fertig
Durchgang 25 fertig
1000 Toots in 25 Durchgängen geladen
Toots in hamburg.txt geschrieben


# Topics extrahieren

In [4]:
from bertopic import BERTopic
from umap import UMAP
from sentence_transformers import SentenceTransformer
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer
import gensim.corpora as corpora
from gensim.models import CoherenceModel
import os

FILE = 'hamburg.txt' # Mit Scraper extrahierte Toots
HASHTAG = "hamburg" # Ohne #

# BERTopic Parameter
EMBEDDING = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
#EMBEDDING = 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2'
#EMBEDDING = 'all-MiniLM-L6-v2'
NR_TOPICS = None        # Topic Anzahl nicht vorgeben
MIN_TOPIC_SIZE = 10     # Topic Mindestgröße
N_NEIGHBORS = 60        # Anzahl Nachbarn
MIN_DIST = 0.05         # Entfernung zwischen Punkten

# Tokenizers Warnung abschalten
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Stopwörterliste erstellen
try:
    stop_words = stopwords.words('german')
except LookupError:
    nltk.download('stopwords')
    stop_words = stopwords.words('german')

stop_words.append(HASHTAG)   # HASHTAG wird als Stopwort eingetragen um in den Topics nicht aufzutauchen
stop_words.extend(stopwords.words('english'))

# Toots aus Datei laden
def load_list_from_file(filename):
    data_list = []
    with open(filename, 'r', encoding='utf-8') as f:
        for line in f:
            data_list.append(line.rstrip('\n'))
    return data_list

# Topic Wörter pro Topic anzeigen
def get_topic_words(topic_model, topic_id, top_n=10):
    words = topic_model.get_topic(topic_id)
    return [word for word, score in words[:top_n]]

docs = load_list_from_file(FILE)    # Datei laden

# Topics mit BERTopic extrahieren
try:
    vectorizer = CountVectorizer(
        stop_words=stop_words,
        min_df=2,              # Wörter ignorieren, die <2 mal vorkommen
        ngram_range=(1, 2),    # Bigramme
        lowercase=True         # Nur Kleinschreibung
    )

    umap_model = UMAP(n_neighbors=N_NEIGHBORS, n_components=5, min_dist=MIN_DIST, metric='cosine', random_state=12)
    embedding_model = SentenceTransformer(EMBEDDING)
    topic_model = BERTopic(
        language="german",
        embedding_model=embedding_model,
        umap_model=umap_model,
        vectorizer_model=vectorizer,
        nr_topics=NR_TOPICS,
        min_topic_size=MIN_TOPIC_SIZE,
        calculate_probabilities=True,
        #verbose=True
    )

    topics, _ = topic_model.fit_transform(docs)

except Exception as e:
    print(f"Fehler bei BERTopic Verarbeitung: {e}")
    exit()

# Coherence Score berechnen
try:
    cleaned_docs = topic_model._preprocess_text(docs)
    analyzer = vectorizer.build_analyzer()
    tokens = [analyzer(doc) for doc in cleaned_docs]
    dictionary = corpora.Dictionary(tokens)
    corpus = [dictionary.doc2bow(token) for token in tokens]
    topics = topic_model.get_topics()
    topics.pop(-1, None)
    topic_words = [
        [word for word, _ in topic_model.get_topic(topic) if word != ""]
        for topic in range(len(set(topics)))]

    coherence_model = CoherenceModel(topics=topic_words,
        texts=tokens,
        corpus=corpus,
        dictionary=dictionary,
        coherence='c_v')

    coherence_score = coherence_model.get_coherence()

except Exception as e:
    print(f"Fehler bei der Coherence Score Berechnung: {e}")
    exit()

# Ausgabe
try:
    topic_info = topic_model.get_topic_info()
    topic_info = topic_info[topic_info['Topic'] != -1]

    print("\n" + "="*35)
    print("            Ergebnis")
    print("="*35)

    print(f"Gesamtanzahl gefundener Topics: {len(topic_info)}")
    print(f"Coherence Score: {coherence_score:.5f}")
    print("Topics:")

    for _, row in topic_info.iterrows():
        topic_id = row['Topic']
        topic_words = get_topic_words(topic_model, topic_id, top_n=10)
        print(f"\nTopic {topic_id + 1} (In {row['Count']} Toots enthalten)")
        print("  ", ", ".join(topic_words))

except Exception as e:
    print(f"Fehler bei der Ausgabe: {e}")
    exit()


            Ergebnis
Gesamtanzahl gefundener Topics: 15
Coherence Score: 0.53101
Topics:

Topic 1 (In 130 Toots enthalten)
   00, 2026, mai 2026, 19, 2026 19, mai, 19 00, 30, hafenklang, juni 2026

Topic 2 (In 95 Toots enthalten)
   bahn, unfall, harburg, gesperrt, straße, hamburger, hamburger abendblatt, abendblatt, hochbahn, auto

Topic 3 (In 52 Toots enthalten)
   polizei, mann, 17, jähriger, wegen, news, hamburger, jährige, hamburger abendblatt, abendblatt

Topic 4 (In 38 Toots enthalten)
   elbphilharmonie, hamburger abendblatt, abendblatt, hamburger, band, konzert, theater, bühne, ensemble, stück

Topic 5 (In 34 Toots enthalten)
   köln, moin, mass, miniatur wunderland, miniatur, wunderland, asphaltsprenger, mai, juli, danke

Topic 6 (In 32 Toots enthalten)
   camp, moorweide, kritik, camp moorweide, frauen, protestcamp, hamburger, ort, pro, antisemitismus

Topic 7 (In 32 Toots enthalten)
   nolympia, olympia, nolympia_hamburg, nolympia olympia, social, norden, umwelt, 000, dosb